# Lab 1 &ndash; Digital ledgers

In this first lab, we'll work on the relationship between the **states** of a digital ledger and its list of **operations**.

Done by,<span style="color: blue">

*   Mishael Natth Viswanathan
*   Boudry Nathan
*   Teophile Pradalier-Lefebvre
</span>

## A) Basic transfer

To keep it simple, we'll assume for now that the states are described as Python dictionaries mapping the names of the variables (strings) to their values (Python integers). For example:

In [ ]:
state = { "Alice": 10, "Bob": 30, "Carol": 42 }

The most basic type of operation is a `Transfer` of a certain number of units (tokens) from one variable to another. A `Transfer` is thus described by:

- a `source` variable name;
- a `target` variable name;
- an integer `amount` to be transferred from `source` to `target`.

For example, that operation could be:

In [ ]:
op = { "op_type": "Transfer", "source": "Alice", "target": "kevin", "amount": 5 }

Such an operation is admissible only if `source` exists in the current state and its value is $\geq$ `amount`, Its effect is to decrease `source` by `amount`, and increase `target` by amount if `target` exists in state (else, add a `target` variable to the state and initialize it to `amount`). That way, we can easily transfer tokens to new users that were not initially present in the chain.

**To do** : write a function `apply_op` that takes as input a state and an operation, checks if the operation is valid then applies it and returns the resulting state. If the operation is invalid, the initial state is returned unchanged.

In [ ]:
state = { "Alice": 10, "Bob": 30, "Carol": 42 }
op = { "op_type": "Transfer", "source": "Alice", "target": "Bob", "amount": 5 }

def apply_op(state,op):

    source = op["source"]
    target = op["target"]
    amount = op["amount"]

    if op["op_type"] == "Transfer":
        if source in state and state[source] >= amount:
            state[source] -= amount
            if target in state:
                state[target] += amount
            else:
                state[target] = amount

    return state

print(apply_op(state,op))

{'Alice': 5, 'Bob': 35, 'Carol': 42}


For example: with the above example you should obtain as resulting state

In [ ]:
{ "Alice": 5, "Bob": 35, "Carol": 42 }

{'Alice': 5, 'Bob': 35, 'Carol': 42}

## B) Keeping balance

Let us add a new type of operation `Init` that has a `name` and a `value` that allows us to initialize a new variable `name` with value `value`. This operation would only be admissible if `name` was not already declared in the current state.

**To do**: implement this new operation by modifying `apply_op` accordingly. Provide also a function `apply_ops` that will apply sequentially a *list* of operation to a starting state.

In [ ]:
state = {}

ops = [
    { "op_type": "Init", "name": "Tibo", "value": 100 },
    { "op_type": "Init", "name": "Nathan", "value": 300 },
    { "op_type": "Init", "name": "Léo", "value": 420 },
    { "op_type": "Transfer", "source": "Tibo", "target": "Nathan", "amount": 50}
]

def apply_op(state,op):
    if op["op_type"] == "Transfer":
        if op["source"] in state and state[op["source"]] >= op["amount"]:
            state[op["source"]] -= op["amount"]
            if op["target"] in state:
                state[op["target"]] += op["amount"]
            else:
                state[op["target"]] = op["amount"]

    elif op["op_type"] == "Init":
        if op["name"] not in state:
            state[op["name"]] = op["value"]

    return state

def apply_ops(state,ops):
    for op in ops:
        state = apply_op(state,op)
    return state

print(apply_ops(state,ops))

{'Tibo': 50, 'Nathan': 350, 'Léo': 420}


For example, the following list of operations gives the same final state as the example of A) (starting from an empty state):

In [ ]:
[
    {" op_type": "Init", "name": "Alice", "value": 10 },
    { "op_type": "Init", "name": "Bob", "value": 30 },
    { "op_type": "Init", "name": "Carol", "value": 42 },
    { "op_type": "Transfer", "source": "Alice", "target": "Bob", "amount": 5}
]

[{' op_type': 'Init', 'name': 'Alice', 'value': 10},
 {'op_type': 'Init', 'name': 'Bob', 'value': 30},
 {'op_type': 'Init', 'name': 'Carol', 'value': 42},
 {'op_type': 'Transfer', 'source': 'Alice', 'target': 'Bob', 'amount': 5}]

What is the final state for the list of operations given in `ledger1.json`? You can import it in Python as below:

In [ ]:
import json

with open("ledger1.json") as file:
    ops = json.load(file)

## C) A larger ledger

Let's now do the same thing with the slightly larger (just over a million operations) `ledger2.json`. What is the resulting state if these operations are applied to an initial empty state?

In [ ]:
import json

with open("ledger2.json") as file:
    ops = json.load(file)

Note that, if we know (or assume) that all operations recorded in the ledger are valid, then we do not need to check validity everytime we apply an operation (thus slightly speeding up the process). Try it by deactivating the verification of the validity of the operations in your `apply_op` function: is the speedup significative?

**Measure standard**

In [ ]:
import time

# Measure Standard
state_std = {}
t0 = time.time()
apply_ops(state_std, ops)
t1 = time.time()
time_std = t1 - t0
print(f"Time with validation: {time_std:.4f} seconds")

Time with validation: 0.8910 seconds


In [ ]:
def apply_op_fast(state, op):
    if op["op_type"] == "Transfer":
        state[op["source"]] -= op["amount"]
        if op["target"] in state:
            state[op["target"]] += op["amount"]
        else:
            state[op["target"]] = op["amount"]
    elif op["op_type"] == "Init":
        state[op["name"]] = op["value"]
    return state

def apply_ops_fast(state, ops):
    for op in ops:
        state = apply_op_fast(state, op)
    return state

In [ ]:

# Measure Fast
state_fast = {}
t0 = time.time()
apply_ops_fast(state_fast, ops)
t1 = time.time()
time_fast = t1 - t0
print(f"Time without validation: {time_fast:.4f} seconds")

print(f"Speedup: {((time_std - time_fast) / time_std) * 100:.2f}%")

Time without validation: 0.6511 seconds
Speedup: 26.92%


We can shorten the process even further if we are only interested in a single variable value by following only the relevant part of the state (namely: the value of this precise variable). How much quicker is it to proceed in this fashion if we only want the final value of, say, variable `c2d55089ad0cfada`?

In [ ]:
import time

target_var = 'c2d55089ad0cfada'
t0 = time.time()

# Track only the target variable
target_value = 0
for op in ops:
    op_type = op["op_type"]
    if op_type == "Init":
        if op["name"] == target_var:
            target_value = op["value"]
    elif op_type == "Transfer":
        if op["source"] == target_var:
            target_value -= op["amount"]
        elif op["target"] == target_var:
            target_value += op["amount"]

t1 = time.time()
time_single = t1 - t0

print(f"Final value of {target_var}: {target_value}")
print(f"Time tracking only one variable: {time_single:.4f} seconds")
if 'time_fast' in globals():
    speedup_vs_fast = ((time_fast - time_single) / time_fast) * 100
    print(f"Speedup compared to full state (no validation): {speedup_vs_fast:.2f}%")
if 'time_std' in globals():
    speedup_vs_std = ((time_std - time_single) / time_std) * 100
    print(f"Speedup compared to full state (with validation): {speedup_vs_std:.2f}%")

Final value of c2d55089ad0cfada: -203764
Time tracking only one variable: 0.3875 seconds
Speedup compared to full state (no validation): 40.49%
Speedup compared to full state (with validation): 56.51%


## D) Custom operations

Add some more functionality to our very basic ledger system by adding and implementing some other operations. Make sure to specify the validity conditions of your operations, and provide a couple of working examples. Some (non-exhaustive!) ideas:

- basic arithmetic operations (add, multiply, divide, ...)
- some less basic arithmetic operations (GCD, sum of divisors, Euler phi function, ...)
- some more advanced arithmetic operations (checking primality, finding factors, ...)
- `End-of-Year Bonus`: all variables are incremented by 1! (or only those that start with the letter `G`)
- `Cleanup`: remove from state all empty accounts (variables with value 0)
- `Snapshot`: ignore previous state and re-initialize all variables to given values (allowing one to "compress" the ledger by forgetting old operations without changing the resulting state)
- implementing a *non-fungible token*: only a single one can be emitted (`InitNFT` operation) and transferred wholly for one variable (user) to another
- ...

### Custom Operations Implementation

We will define the following custom operations with their validity rules:
- **`Add`**: Admissible if the `name` exists. Increases `name`'s balance by `value`.
- **`Multiply`**: Admissible if the `name` exists and `factor` is an integer. Multiplies the balance by `factor`.
- **`Bonus`**: Admissible always. Increments either all accounts or only those starting with a specified prefix (e.g., `'G'`) by a given `amount`.
- **`Cleanup`**: Admissible always. Deletes any variables/accounts from the state that currently hold a value of `0`.

In [ ]:
def apply_op_custom(state, op, nft_registry=None):
    if nft_registry is None:
        # Simple shared/persistent NFT tracking dict: token_id -> owner
        nft_registry = {}

    op_type = op["op_type"]

    if op_type == "Init":
        if op["name"] not in state:
            state[op["name"]] = op["value"]

    elif op_type == "Transfer":
        if op["source"] in state and state[op["source"]] >= op["amount"]:
            state[op["source"]] -= op["amount"]
            state[op["target"]] = state.get(op["target"], 0) + op["amount"]

    elif op_type == "Add":
        # Validity: Account must exist
        if op["name"] in state:
            state[op["name"]] += op["value"]

    elif op_type == "Multiply":
        # Validity: Account must exist
        if op["name"] in state:
            state[op["name"]] *= op["factor"]

    elif op_type == "Bonus":
        # Validity: Always admissible
        prefix = op.get("prefix", "")
        amount = op.get("amount", 1)
        for name in list(state.keys()):
            if name.startswith(prefix):
                state[name] += amount

    elif op_type == "Cleanup":
        # Validity: Always admissible
        for name in list(state.keys()):
            if state[name] == 0:
                del state[name]

    elif op_type == "Snapshot":
        # Re-initializes state with the given values entirely
        state.clear()
        for name, val in op["values"].items():
            state[name] = val

    elif op_type == "InitNFT":
        # Only one NFT with a given ID can be initialized/minted globally
        token_id = op["token_id"]
        owner = op["owner"]
        if token_id not in nft_registry and owner in state:
            nft_registry[token_id] = owner

    elif op_type == "TransferNFT":
        # Can only transfer if the source actually owns this token_id
        token_id = op["token_id"]
        source = op["source"]
        target = op["target"]
        if nft_registry.get(token_id) == source and target in state:
            nft_registry[token_id] = target

    return state, nft_registry

def apply_ops_custom(state, ops):
    nft_registry = {}
    for op in ops:
        state, nft_registry = apply_op_custom(state, op, nft_registry)
    return state, nft_registry

Let us verify our custom ledger operations with a step-by-step test execution.

In [ ]:
demo_state = {}
demo_ops = [

    # Arithmetic: Add 50 to Geralt
    { "op_type": "Add", "name": "Geralt", "value": 50 },
    # End-of-Year Bonus: increment only those starting with 'G' by 10
    { "op_type": "Bonus", "prefix": "G", "amount": 10 },
    # Cleanup: remove Jaskier because his balance is 0
    { "op_type": "Cleanup" },
    # Mint an NFT "Medallion" to Geralt
    { "op_type": "InitNFT", "token_id": "Medallion", "owner": "Geralt" },
    # Transfer ownership of "Medallion" to Yennefer
    { "op_type": "TransferNFT", "token_id": "Medallion", "source": "Geralt", "target": "Yennefer" },
    # Snapshot compression: reset ledger to a compressed state representation
    { "op_type": "Snapshot", "values": { "Geralt": 160, "Yennefer": 300, "Ciri": 500 } },
    # Mint another NFT after the snapshot
    { "op_type": "InitNFT", "token_id": "ElderBlood", "owner": "Ciri" }
]

final_state, nfts = apply_ops_custom(demo_state, demo_ops)
print("Final Ledgers State:")
display(final_state)
print("NFT Ownership Registry:")
display(nfts)

Final Ledgers State:


{'Geralt': 160, 'Yennefer': 300, 'Ciri': 500}

NFT Ownership Registry:


{'ElderBlood': 'Ciri'}